### Fichier utilisé pour fusionner les données classifiées par les modèles de NLP sur les interactions structurées
Produit `struct_annotated_interactions`

In [ ]:
import pandas as pd

1. Fusionner les messages toxiques sur les messages politiques

In [ ]:
flat_political = pd.read_csv('../clean_data/flat_political_interactions.csv')

flat_toxic = pd.read_csv('../clean_data/flat_toxic_interactions.csv', index_col=0)
flat_toxic = flat_toxic.sort_values("toxicity_level", ascending=False).drop_duplicates(subset="text", keep="first")
flat_toxic['violent'] = (flat_toxic['toxicity_level'] >= flat_toxic['toxicity_level'].quantile(0.75)).astype(int)
flat_toxic = flat_toxic.drop(columns=['toxicity_level'])

In [ ]:
df_merge = pd.merge(flat_political, flat_toxic, on='text', how='left', indicator=True)
print(df_merge['_merge'].value_counts())
# Tous les messages dans la classification toxique sont inclus dans politique
df_merge = df_merge.drop(columns=['_merge'])

2. Fusionner les messages gauche-droite sur les messages politiques

In [ ]:
flat_left_right = pd.read_csv('../clean_data/flat_left_right_interactions.csv')
df_merge = pd.merge(df_merge, flat_left_right, on='text', how='left', indicator=True)
print(df_merge['_merge'].value_counts())
# Tous les messages classifiés gauche-droite sont dans politique
df_merge = df_merge.drop(columns=['_merge'])

3. Fusionner les messages politiques dans tous les messages

In [ ]:
flat_interactions = pd.read_csv('../clean_data/flat_all_interactions.csv')
df_merge = pd.merge(flat_interactions, df_merge, on='text', how='left', indicator='political')

In [ ]:
print(df_merge['political'].value_counts())
# Tous les messages politiques sont dans le jeu de message plat
df_merge['political'] = df_merge['political'].replace({
    'left_only': 0,
    'both':1
    }).cat.remove_unused_categories()
print(df_merge['political'].value_counts())

4. Fusionner tous les messages plats dans les interactions structurées

In [ ]:
struct_interactions = pd.read_csv('../clean_data/struc_all_interactions.csv')
struct_interactions = struct_interactions.rename(columns={
    'text':'user_text',
    'text_inter':'inter_text'
})
struct_interactions = struct_interactions.drop(columns=['pfp_inter', 'after', 'date_ins'])

In [ ]:
struct_panel = pd.merge(struct_interactions, df_merge, left_on='user_text', right_on='text', how='left')
struct_panel = struct_panel.rename(columns={
    'violent':'user_violent', 
    'left_right':'user_left_right', 
    'political':'user_political'
    })
struct_panel = struct_panel.drop(columns=['text'])

In [ ]:
struct_panel = pd.merge(struct_panel, df_merge, left_on='inter_text', right_on='text', how='left')
struct_panel = struct_panel.rename(columns={
    'violent':'inter_violent', 
    'left_right':'inter_left_right', 
    'political':'inter_political'
    })
struct_panel = struct_panel.drop(columns=['text'])

In [ ]:
struct_panel = pd.merge(struct_panel, df_merge, left_on='post_text', right_on='text', how='left')
struct_panel = struct_panel.rename(columns={
    'violent':'post_violent', 
    'left_right':'post_left_right', 
    'political':'post_political'
    })
struct_panel = struct_panel.drop(columns=['text'])

5. Exporter les interactions structurées classifiées

In [ ]:
struct_panel.to_csv('../clean_data/struct_classified_interactions.csv', index=False)